# Open Health Risk Engine

This notebook is a Kaggle-ready walkthrough of the project behind the Open Health Risk Engine repository.

It focuses on four things:

- loading the saved evaluation artifacts
- reviewing model comparison and calibration outputs
- inspecting subgroup metrics with bootstrap confidence intervals
- scoring a few example profiles with the deployed inference wrapper

Publishing note:
Attach the repository snapshot as a Kaggle dataset, or open this notebook directly from the repo after uploading the project files to Kaggle.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display


def resolve_root() -> Path:
    candidates = [
        Path("/kaggle/input/open-health-risk-engine"),
        Path("/kaggle/input/open-health-risk-engine/open-health-risk-engine"),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    for candidate in candidates:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repo root. Attach the project files as a Kaggle dataset or open the notebook from the repo."
    )


ROOT = resolve_root()
sys.path.insert(0, str(ROOT))

from src.predict_risk import RiskPredictor

ROOT

In [ ]:
test_results = pd.read_csv(ROOT / "models" / "test_results.csv")
cv_results = pd.read_csv(ROOT / "models" / "cv_results.csv")
calibrated_summary = pd.read_csv(ROOT / "models" / "calibrated_summary.csv")
error_summary = pd.read_csv(ROOT / "models" / "error_outcome_summary.csv")

display(cv_results)
display(test_results)
display(calibrated_summary)
display(
    error_summary[
        [
            "error_outcome",
            "n",
            "share_of_test",
            "mean_predicted_probability",
            "mean_phq9_score",
        ]
    ]
)

In [ ]:
subgroup_metrics = pd.read_csv(ROOT / "models" / "subgroup_metrics.csv")
subgroup_view = subgroup_metrics[
    [
        "subgroup_type",
        "subgroup",
        "n",
        "auc_roc",
        "auc_roc_ci_low",
        "auc_roc_ci_high",
        "recall",
        "recall_ci_low",
        "recall_ci_high",
        "f1",
        "f1_ci_low",
        "f1_ci_high",
    ]
]

display(subgroup_view.sort_values(["subgroup_type", "subgroup"]).reset_index(drop=True))

In [ ]:
predictor = RiskPredictor(ROOT / "models" / "best_model.joblib")

scenario_rows = [
    {
        "profile": "Active profile",
        "inputs": {
            "age": 30,
            "sex_female": 0,
            "poverty_ratio": 3.5,
            "met_min_week": 900,
            "sleep_hours": 8.0,
            "sleep_trouble": 0,
            "bmi": 23.0,
            "drinks_per_week": 2,
            "education": 5,
            "race_eth": 3,
        },
    },
    {
        "profile": "Higher-risk lifestyle profile",
        "inputs": {
            "age": 45,
            "sex_female": 1,
            "poverty_ratio": 0.8,
            "met_min_week": 0,
            "sleep_hours": 5.0,
            "sleep_trouble": 1,
            "bmi": 32.0,
            "drinks_per_week": 14,
            "education": 2,
            "race_eth": 4,
        },
    },
    {
        "profile": "Mixed-signal profile",
        "inputs": {
            "age": 52,
            "sex_female": 0,
            "poverty_ratio": 2.1,
            "met_min_week": 250,
            "sleep_hours": 6.0,
            "sleep_trouble": 1,
            "bmi": 28.5,
            "drinks_per_week": 6,
            "education": 3,
            "race_eth": 3,
        },
    },
]

scored_rows = []
for scenario in scenario_rows:
    result = predictor.predict(scenario["inputs"])
    scored_rows.append(
        {
            "profile": scenario["profile"],
            "risk_score": result["risk_score"],
            "risk_label": result["risk_label"],
            "phq9_estimate": result["phq9_estimate"],
            "top_factors": ", ".join(factor["feature"] for factor in result["top_factors"][:3]),
        }
    )

pd.DataFrame(scored_rows)

In [ ]:
figure_names = [
    "app_walkthrough.png",
    "roc_curves.png",
    "shap_summary.png",
    "calibration_curve_random_forest.png",
]

for figure_name in figure_names:
    figure_path = ROOT / "figures" / figure_name
    if figure_path.exists():
        print(figure_name)
        display(Image(filename=str(figure_path), width=900))

## Kaggle publishing checklist

1. Upload the repo snapshot or attach it as a Kaggle dataset.
2. Keep the `models/` and `figures/` folders available so the notebook can load artifacts immediately.
3. If you want a full retraining notebook later, reuse the repo scripts under `src/` and swap in the NHANES raw files as Kaggle dataset assets.
4. Add the live app and GitHub repo links in the Kaggle notebook description when publishing.